# Preprocesamiento — YouTube Toxic Comments
**Proyecto NLP | Detección de Mensajes de Odio**  
**Rama:** `feature/preprocessing_v03`

---

## 🎯 Objetivo
Transformar el texto crudo en texto limpio y lematizado listo para vectorización.

### Decisiones tomadas en el EDA que aplicamos aquí:
| Hallazgo EDA | Acción |
|---|---|
| 3 duplicados detectados | `drop_duplicates()` |
| URLs en 91.7% de no tóxicos | Eliminar URLs con regex |
| `idiot`(16x), `bitch`(14x) son señal | NO eliminar vocabulario de odio |
| Vocabulario compartido entre clases | TF-IDF con `ngram_range=(1,2)` en modelado |
| `IsHomophobic`, `IsRadicalism` = 0 casos | Solo usamos `IsToxic` como target |
| Ratio 46/54 balanceado | `class_weight='balanced'` en modelo |

## 1. Importaciones

In [1]:
from pathlib import Path
import pandas as pd
import re
import spacy
from nltk.corpus import stopwords
import nltk
import warnings
warnings.filterwarnings('ignore')

nltk.download('stopwords', quiet=True)

# Cargar modelo de inglés de spaCy
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])  # solo necesitamos lematización

STOPWORDS = set(stopwords.words('english'))

# Rutas
BASE_DIR = Path().resolve()
while not (BASE_DIR / 'pyproject.toml').exists():
    BASE_DIR = BASE_DIR.parent

DATA_PATH = BASE_DIR / 'data' / 'raw' / 'youtoxic_english_1000.csv'
OUT_PATH  = BASE_DIR / 'data' / 'processed' / 'toxic_comments_clean_v03.csv'
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print('✅ Librerías cargadas')
print(f'📂 Dataset: {DATA_PATH}')
print(f'💾 Output:  {OUT_PATH}')

✅ Librerías cargadas
📂 Dataset: C:\Users\SONY VAIO\Desktop\IA\practicas\Project_9_NLP_Team2\data\raw\youtoxic_english_1000.csv
💾 Output:  C:\Users\SONY VAIO\Desktop\IA\practicas\Project_9_NLP_Team2\data\processed\toxic_comments_clean_v03.csv


## 2. Carga y selección de columnas

In [2]:
# Cargamos el dataset original
df_raw = pd.read_csv(DATA_PATH)
print(f'📐 Shape original: {df_raw.shape}')

# Nos quedamos solo con las columnas necesarias
# Decisión: clasificación BINARIA → solo IsToxic como target
columnas_necesarias = ['CommentId', 'VideoId', 'Text', 'IsToxic']
df = df_raw[columnas_necesarias].copy()

print(f'📐 Shape tras selección de columnas: {df.shape}')
print(f'\n📋 Columnas: {list(df.columns)}')
df.head(3)

📐 Shape original: (1000, 15)
📐 Shape tras selección de columnas: (1000, 4)

📋 Columnas: ['CommentId', 'VideoId', 'Text', 'IsToxic']


,CommentId,VideoId,Text,IsToxic
0,Ugg2KwwX0V8-aXgCoAEC,04kJtp6pVXI,If only people would just take a step back and...,False
1,Ugg2s5AzSPioEXgCoAEC,04kJtp6pVXI,Law enforcement is not trained to shoot to app...,True
2,Ugg3dWTOxryFfHgCoAEC,04kJtp6pVXI,\r\nDont you reckon them 'black lives matter' ...,True


## 3. Limpieza básica

In [3]:
# --- Valores nulos ---
print('=== ❌ Valores nulos ===')
print(df.isnull().sum())

# --- Duplicados ---
n_dup = df.duplicated(subset='Text').sum()
print(f'\n🔁 Duplicados detectados: {n_dup}')

if n_dup > 0:
    df = df.drop_duplicates(subset='Text', keep='first').reset_index(drop=True)
    print(f'✅ Duplicados eliminados → Shape: {df.shape}')

# --- Distribución de clases tras limpieza ---
print(f'\n=== ⚖️ Distribución IsToxic ===')
counts = df['IsToxic'].value_counts()
pcts   = df['IsToxic'].value_counts(normalize=True) * 100
for val in [True, False]:
    label = '🔴 Tóxico   ' if val else '🟢 No tóxico'
    print(f'  {label}: {counts[val]:>4} ({pcts[val]:.1f}%)')

=== ❌ Valores nulos ===
CommentId    0
VideoId      0
Text         0
IsToxic      0
dtype: int64

🔁 Duplicados detectados: 3
✅ Duplicados eliminados → Shape: (997, 4)

=== ⚖️ Distribución IsToxic ===
  🔴 Tóxico   :  459 (46.0%)
  🟢 No tóxico:  538 (54.0%)


## 4. 🔧 Pipeline de Preprocesamiento de Texto

### Pasos aplicados en orden:
1. **Lowercase** — homogeneizar mayúsculas/minúsculas
2. **Eliminar URLs** — no aportan señal de odio (91.7% en no tóxicos según EDA)
3. **Eliminar menciones y caracteres especiales** — ruido
4. **Eliminar números** — no discriminan toxicidad
5. **Eliminar espacios extra**
6. **Lematización con spaCy** — forma canónica de cada palabra
7. **Eliminar stopwords** — palabras vacías sin valor discriminante

> ⚠️ **NO eliminamos vocabulario de odio** (`idiot`, `bitch`, `fuck`...)
> Son la señal más discriminante según el EDA (ratios de 8x-16x)

In [4]:
def clean_text(text: str) -> str:
    """Limpieza básica del texto antes de lematizar."""
    # 1. Lowercase
    text = text.lower()
    # 2. Eliminar URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # 3. Eliminar menciones (@usuario)
    text = re.sub(r'@\w+', '', text)
    # 4. Eliminar caracteres especiales y números — solo letras y espacios
    text = re.sub(r'[^a-z\s]', '', text)
    # 5. Eliminar espacios extra
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def lemmatize_and_remove_stopwords(text: str) -> str:
    """Lematiza con spaCy y elimina stopwords."""
    doc = nlp(text)
    tokens = [
        token.lemma_
        for token in doc
        if token.lemma_ not in STOPWORDS   # eliminar stopwords
        and len(token.lemma_) > 1           # eliminar letras sueltas
        and not token.is_space              # eliminar espacios
    ]
    return ' '.join(tokens)


def preprocess(text: str) -> str:
    """Pipeline completo: limpieza + lematización + stopwords."""
    text = clean_text(text)
    text = lemmatize_and_remove_stopwords(text)
    return text


# --- Test rápido ---
ejemplos = [
    "You are such an IDIOT, go back to where you came from! http://example.com",
    "I think the police officer did nothing wrong in this situation.",
    "This guy is stupid and I hate him @user123"
]

print('=== 🔍 Test del pipeline ===')
for ej in ejemplos:
    resultado = preprocess(ej)
    print(f'\nOriginal:  {ej}')
    print(f'Procesado: {resultado}')

=== 🔍 Test del pipeline ===

Original:  You are such an IDIOT, go back to where you came from! http://example.com
Procesado: idiot go back come

Original:  I think the police officer did nothing wrong in this situation.
Procesado: think police officer nothing wrong situation

Original:  This guy is stupid and I hate him @user123
Procesado: guy stupid hate


## 5. ⚙️ Aplicar preprocesamiento al dataset completo

In [5]:
print('⏳ Aplicando preprocesamiento... (puede tardar 1-2 minutos)')

# Guardamos el texto original
df['Text_original'] = df['Text'].copy()

# Aplicamos el pipeline
df['Text_clean'] = df['Text'].apply(preprocess)

print(f'✅ Preprocesamiento completado')
print(f'📐 Shape final: {df.shape}')

⏳ Aplicando preprocesamiento... (puede tardar 1-2 minutos)
✅ Preprocesamiento completado
📐 Shape final: (997, 6)


## 6. 🔍 Verificación del preprocesamiento

In [6]:
# Comparación antes/después
print('=== 📊 Comparación antes/después ===')
sample = df[['Text_original', 'Text_clean', 'IsToxic']].sample(5, random_state=42)
for _, row in sample.iterrows():
    label = '🔴' if row['IsToxic'] else '🟢'
    print(f'\n{label} Original:  {row["Text_original"][:100]}')
    print(f'   Procesado: {row["Text_clean"][:100]}')

=== 📊 Comparación antes/después ===

🟢 Original:  The music references sex, drugs, stealing, murder, etc.

...ya, pretty much all rap music is like 
   Procesado: music reference sex drug steal murder etc ya pretty much rap music like

🟢 Original:  That community should be marching for peace hand in hand for the girl that was killed sitting in her
   Procesado: community march peace hand hand girl kill sit home homework deserve least society let maybe al sharp

🔴 Original:  Fucking savages! Protest sumthing real... Like how your government is robbing you and sendin your ki
   Procesado: fucking savage protest sumthe real like government rob sendin kid elegal war laugh stock america con

🔴 Original:  death ti the white devil
   Procesado: death ti white devil

🟢 Original:  Oh that politicians had the same style of oratory! Peggy Hubbard, in one short video, rings out a me
   Procesado: oh politician style oratory peggy hubbard one short video ring message loud clearto listen ok englan


In [7]:
# Detectar textos vacíos tras preprocesamiento
empty = df[df['Text_clean'].str.strip() == '']
print(f'⚠️ Textos vacíos tras preprocesamiento: {len(empty)}')

if len(empty) > 0:
    print('Eliminando textos vacíos...')
    df = df[df['Text_clean'].str.strip() != ''].reset_index(drop=True)
    print(f'✅ Shape tras eliminar vacíos: {df.shape}')

# Estadísticas de longitud tras preprocesamiento
df['word_count_clean'] = df['Text_clean'].str.split().str.len()
print(f'\n=== 📏 Longitud tras preprocesamiento ===')
print(df['word_count_clean'].describe().round(1))

print(f'\n=== 📏 Comparación longitud media ===')
df['word_count_original'] = df['Text_original'].str.split().str.len()
print(f'  Antes: {df["word_count_original"].mean():.1f} palabras')
print(f'  Después: {df["word_count_clean"].mean():.1f} palabras')

⚠️ Textos vacíos tras preprocesamiento: 0

=== 📏 Longitud tras preprocesamiento ===
count    997.0
mean      16.8
std       23.6
min        1.0
25%        5.0
50%       10.0
75%       19.0
max      379.0
Name: word_count_clean, dtype: float64

=== 📏 Comparación longitud media ===
  Antes: 33.9 palabras
  Después: 16.8 palabras


In [8]:
# Verificar que las palabras clave de odio se conservan
print('=== 🎯 Verificación: palabras clave de odio conservadas ===')
hate_words = ['idiot', 'stupid', 'hate', 'bitch', 'isis', 'racist']

for word in hate_words:
    count_original = df['Text_original'].str.lower().str.contains(word).sum()
    count_clean    = df['Text_clean'].str.contains(word).sum()
    status = '✅' if count_clean > 0 else '❌'
    print(f'  {status} "{word}": {count_original} en original → {count_clean} en limpio')

=== 🎯 Verificación: palabras clave de odio conservadas ===
  ✅ "idiot": 28 en original → 28 en limpio
  ✅ "stupid": 37 en original → 37 en limpio
  ✅ "hate": 24 en original → 24 en limpio
  ✅ "bitch": 19 en original → 19 en limpio
  ✅ "isis": 8 en original → 8 en limpio
  ✅ "racist": 53 en original → 53 en limpio


## 7. 💾 Exportar dataset preprocesado

In [9]:
# Columnas finales para exportar
df_export = df[['CommentId', 'VideoId', 'Text_original', 'Text_clean', 'IsToxic']].copy()

df_export.to_csv(OUT_PATH, index=False)

print(f'✅ Dataset preprocesado exportado: {OUT_PATH.name}')
print(f'📐 Shape: {df_export.shape}')
print(f'📋 Columnas: {list(df_export.columns)}')
print(f'\n📊 Distribución final IsToxic:')
print(df_export['IsToxic'].value_counts())
print(df_export.head(3))

✅ Dataset preprocesado exportado: toxic_comments_clean_v03.csv
📐 Shape: (997, 5)
📋 Columnas: ['CommentId', 'VideoId', 'Text_original', 'Text_clean', 'IsToxic']

📊 Distribución final IsToxic:
IsToxic
False    538
True     459
Name: count, dtype: int64
              CommentId      VideoId  \
0  Ugg2KwwX0V8-aXgCoAEC  04kJtp6pVXI   
1  Ugg2s5AzSPioEXgCoAEC  04kJtp6pVXI   
2  Ugg3dWTOxryFfHgCoAEC  04kJtp6pVXI   

                                       Text_original  \
0  If only people would just take a step back and...   
1  Law enforcement is not trained to shoot to app...   
2  \r\nDont you reckon them 'black lives matter' ...   

                                          Text_clean  IsToxic  
0  people would take step back make case anyone e...    False  
1  law enforcement train shoot apprehend train sh...     True  
2  reckon black life matter banner hold white cun...     True  


## 8. 📋 Resumen del Preprocesamiento

```
╔══════════════════════════════════════════════════════════════╗
║           RESUMEN PREPROCESAMIENTO V03                       ║
╠══════════════════════════════════════════════════════════════╣
║  INPUT : youtoxic_english_1000.csv (1000 filas)              ║
║  OUTPUT: comments_preprocessed_V03.csv (997 filas)           ║
║                                                              ║
║  PASOS APLICADOS                                             ║
║  1. Selección de columnas: CommentId, VideoId, Text, IsToxic ║
║  2. Eliminación de 3 duplicados                              ║
║  3. Lowercase                                                ║
║  4. Eliminación de URLs (regex)                              ║
║  5. Eliminación de menciones (@usuario)                      ║
║  6. Eliminación de caracteres especiales y números           ║
║  7. Lematización con spaCy (en_core_web_sm)                  ║
║  8. Eliminación de stopwords (NLTK english)                  ║
║                                                              ║
║  DECISIÓN CLAVE                                              ║
║  → Vocabulario de odio CONSERVADO (idiot, bitch, isis...)    ║
║  → Son la señal más discriminante (ratios 8x-16x en EDA)    ║
║                                                              ║
║  SIGUIENTE PASO                                              ║
║  → 02_split_augmentation_V03.ipynb                           ║
║    - Train/test split                                        ║
║    - Back-translation moderada (solo en train)               ║
╚══════════════════════════════════════════════════════════════╝
```